#**Introduction to RoBERTa**

1. RoBERTa is a reimplementation of BERT with some changes in the key
   hyperparameters and minor embedding tweaks. It uses a byte-level BPE as a tokenizer and a different pretraining scheme.
2. RoBERTa is trained for longer sequences i.e. the number of iterations
   is increased from 100K to 300K and then further to 500K.
3. RoBERTa uses larger byte-level BPE vocabulary with 50K subword units instead
   of character-level BPE vocabulary of size 30K used in BERT.
4. In the Masked Language Model (MLM) training objective, RoBERTa employs dynamic masking to generate the masking pattern every time a sequence is fed to the model.
5. RoBERTa doesn’t use token_type_ids, and we don’t need to define which token
   belongs to which segment. Only separate segments with the separation token tokenizer.sep_token (or ).
6. Larger mini-batches and learning rates are used in RoBERTa’s training.
7. NSP is removed from its objective.

## Uncomment below lines , only require to mount notebook to a given directory and persist the model in **google** drive

In [ ]:
# mounting the drive
# from google.colab import drive
# drive.mount('/content/drive/')

In [ ]:
# # Path to the directory where this model will be saved. If you want to save it to another path then link it with google drive.
# cd  drive/MyDrive/Colab Notebooks

In [ ]:
# Installing the required libraries
!pip install transformers[torch]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
# Accelerate is a library that enables the same PyTorch code to be run across any distributed configuration.
!pip install accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.3.0
    Uninstalling accelerate-1.3.0:
      Successfully uninstalled accelerate-1.3.0


In [ ]:
# Cloning the dataset tweeteval repo here
!git clone https://github.com/cardiffnlp/tweeteval /tmp/tweeteval

# Tweets --> Hate Speech Tweets

Cloning into '/tmp/tweeteval'...
remote: Enumerating objects: 370, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 370 (delta 13), reused 3 (delta 1), pack-reused 354 (from 1)
Receiving objects: 100% (370/370), 8.49 MiB | 18.90 MiB/s, done.
Resolving deltas: 100% (122/122), done.


In [ ]:
#1.
# defining the roberta base model here
from transformers import pipeline

roberta_base_model = pipeline(
                              "fill-mask", # MLM
                              model = "roberta-base",
                              tokenizer = "roberta-base" # BBPE-based
                             )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
pipeline('Send these <mask> back!')

In [ ]:
#2.
# function to predict masked token in sentence
def predict_mask_token(model, sentence):
  predictions = model(sentence)
  for prediction in predictions:
    print(prediction['sequence'].strip('<s>').strip('</s>'), end='\t--- ')
    print(f"{round(100*prediction['score'],2)}% confidence")

# pipeline('Send these <mask> back!')
# Output -->
# <s>  Send these people back! </s>   0.562
# <s>  Send these Mexicans back! </s>  0.31
# <s>  Send these Indians back! </s>   0.054
# 5 such sentences

In [ ]:
#3.
# function call with parameters roberta base model and sentence with mask token
predict_mask_token(roberta_base_model, "Send these <mask> back!")

Send these pictures back!	--- 16.93% confidence
Send these photos back!	--- 11.82% confidence
Send these emails back!	--- 7.12% confidence
Send these letters back!	--- 4.89% confidence
Send these images back!	--- 4.8% confidence


In [ ]:
#4.
predict_mask_token(roberta_base_model, "Elon Musk is the founder of <mask>")

Elon Musk is the founder of Tesla	--- 72.28% confidence
Elon Musk is the founder of SpaceX	--- 26.83% confidence
Elon Musk is the founder of PayPal	--- 0.75% confidence
Elon Musk is the founder of Twitter	--- 0.06% confidence
Elon Musk is the founder of Facebook	--- 0.02% confidence


**Configuring, tokenizing, training & saving the model**

Preparation of dataset(i.e train data converted to tokens with a block size of 512) for training model.

In [ ]:
# MLM task
# Text file --> Dataset --> Tweets [Line by Line]
# Function --> 512 tokens at a time, 15% tokens randomly masked, ... Get the data Line by Line [Data Collator]

In [ ]:
import transformers

In [ ]:
#5.
from transformers import RobertaTokenizer, RobertaForMaskedLM
# constructs a RoBERTa BPE tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
# configuring the model and loading the model weights
model = RobertaForMaskedLM.from_pretrained('roberta-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
#6.
from transformers import LineByLineTextDataset
# Preparing the dataset for training model
dataset = LineByLineTextDataset(
    tokenizer=tokenizer, # for converting train data into tokens
    file_path="/tmp/tweeteval/datasets/hate/train_text.txt", # path where train data file exists
    block_size=512,
)

# T1  --> tokens of T1 [512 from each line]
# T2 --> tokens of T2

/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:119: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


Here first we will understand-

**What is Data Collator and what it does?**

Data collators are objects that will form a batch by using a list of dataset elements as input. These elements are of the same type as the elements of train_dataset or eval_dataset.
To be able to build batches, data collators may apply some processing (like padding). **DataCollatorForLanguageModeling** also apply some random data augmentation (like random masking) on the formed batch.

Here we will make use of **DataCollatorForLanguageModeling** function for masking randomly 15% of the tokens for MLM task. Parameters are

**tokenizer-** The tokenizer used for encoding the data,

**mlm-** Whether or not to use masked language modeling. If set to False, the labels are the same as the inputs with the padding tokens ignored (by setting them to -100). Otherwise, the labels are -100 for non-masked tokens and the value to predict for the masked token.

**mlm_probability-** The probability with which to (randomly) mask tokens in the input, when mlm is set to True.

In [ ]:
#7.
# importing the library DataCollatorForLanguageModeling
from transformers import DataCollatorForLanguageModeling
# For Mask Language Modelling(MLM) task
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15 # for masking randomly 15% of the tokens for MLM task
)

# dataset -->
# [512 tokens from T1]
# [512 tokens from T2]
# ...


# Trainer will take some Training Argument
# Training Arguments: dataset, Data collator (data_collator)


**Note-**

 **Difference between RoBERTa base model and RoBERTa retrained model?**

 RoBERTa base model is very generic model and when we are training this model on a new database then after getting trained it is RoBERTa retrained model.

In [ ]:
#8.
# training the model
from transformers import Trainer, TrainingArguments

# initalizing the training arguments
training_args = TrainingArguments(
    output_dir="./roberta-retrained",  # path where roberta retrained model will be saved.
    overwrite_output_dir=True, # permission for overwriting the output directory
    num_train_epochs=1, # num_train_epochs is a hyperparameter that defines the number of times the learning algorithm will work through the entire training dataset.
    per_device_train_batch_size=48, # The batch size per GPU/TPU core/CPU for training.
    seed=1 # Random seed for initialization. This is optional and defaults to 42.
)

# Passing the model, training arguments, data collator(for masking tokens), dataset to trainer function for training the model
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset # Tokenized dataset
)

In [ ]:
#9.
# model is getting trained here
trainer.train()
# saving the model
trainer.save_model("roberta-retrained/model")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

In [ ]:
#10.
# loading the trained model
roberta_retrained_model = pipeline(
                                    "fill-mask",
                                    model="roberta-retrained/model",
                                    tokenizer="roberta-base"
                                )

In [ ]:
#11.
# Predicting the masked token in sentence with trained model by function call
predict_mask_token(roberta_retrained_model, "Send these <mask> back!")

# # Output -->
# <s>  Send these people back! </s>   0.562
# <s>  Send these Mexicans back! </s>  0.31
# <s>  Send these Indians back! </s>   0.054
# 5 such sentences

In [ ]:
predict_mask_token(roberta_retrained_model, "I hate watching <mask> sports")

In [ ]:
predict_mask_token(roberta_base_model, "I hate watching <mask> sports")

In [ ]:
predict_mask_token(roberta_retrained_model, "Hello I'm a <mask> model.")

In [ ]:
predict_mask_token(roberta_base_model, "Hello I'm a <mask> model.")

In [ ]:
predict_mask_token(roberta_retrained_model, "The man worked as a <mask>.")

In [ ]:
predict_mask_token(roberta_base_model, "The man worked as a <mask>.")

In [ ]:
# doc   ->>>  similar to which of the other docs in your database?

# embedding of these docs?
# Sentence Transformers  -->  1 embedding from the cls token
# Big Doc --> Create chunks of this big doc -->

# t1, t2, ....... t1000
# Chunk1: t1,   t512
# Chunk2: t430, ... t(430 + 512 -1)
# Chunk3:

# for each chunk, get the embedding [Discriminative model] -->
# put it to your vector database

# poses a new doc [overlapping chunks] --> embeddings --> which similar embeddings exist in your database
# get those corresponding chunks

# feed these chunks to your LLM [generative] and ask it to summarize with a prompt!